# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

**Selected document:** [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF).

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
import tempfile
import requests
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

pdf_response = requests.get(PDF_URL, timeout=30)
pdf_response.raise_for_status()

with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp_pdf:
    tmp_pdf.write(pdf_response.content)
    tmp_pdf_path = tmp_pdf.name

loader = PyPDFLoader(tmp_pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages, {len(document_text)} characters.")

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from utils.clients import get_client
from pydantic import BaseModel, Field
import os

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = "gpt-4o-mini"
client = get_client()


class SummaryContent(BaseModel):
    """Fields produced directly by the model."""
    Author: str = Field(description="Author(s) or issuing organization of the source document.")
    Title: str = Field(description="Title of the source document.")
    Relevance: str = Field(description="One paragraph on why this article is relevant for an AI professional's development.")
    Summary: str = Field(description="Concise summary written in Victorian English, no longer than 1000 tokens.")
    Tone: str = Field(description="Name of the tone/style used to write the summary.")


class SummaryResult(SummaryContent):
    """Final reported object: model fields plus token usage pulled from the response object."""
    InputTokens: int
    OutputTokens: int

In [ ]:
INSTRUCTIONS = """
You are an expert analyst who produces structured summaries of professional and business articles.

Write the Summary field entirely in Victorian English: formal 19th-century diction, elaborate
sentence structure, and period-appropriate vocabulary (e.g. "one must observe", "it is with
great interest that..."), while remaining fully accurate to the source material.

Populate every field of the requested schema:
- Author: the author(s) or issuing organization of the piece.
- Title: the title of the piece.
- Relevance: one paragraph explaining why this article is relevant to an AI professional's
  professional development.
- Summary: a concise, accurate summary in Victorian English, no longer than 1000 tokens.
- Tone: name the tone/style used ("Victorian English").
"""

USER_PROMPT = """
Read the following document and produce the requested structured summary.

<document>
{document}
</document>
"""

In [ ]:
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {"role": "user", "content": USER_PROMPT.format(document=document_text)},
    ],
    text_format=SummaryContent,
)

summary_content = response.output_parsed

summary_result = SummaryResult(
    **summary_content.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)

from IPython.display import Markdown, display

display(Markdown(
    f"**Author:** {summary_result.Author}  \n"
    f"**Title:** {summary_result.Title}  \n"
    f"**Tone:** {summary_result.Tone}  \n"
    f"**Input/Output tokens:** {summary_result.InputTokens} / {summary_result.OutputTokens}"
))
display(Markdown(f"**Relevance**\n\n{summary_result.Relevance}"))
display(Markdown(f"**Summary**\n\n{summary_result.Summary}"))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
JUDGE_MODEL = "gpt-4o-mini"

if USE_GATEWAY:
    judge_model = GPTModel(
        model=JUDGE_MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    judge_model = GPTModel(model=JUDGE_MODEL, temperature=0)

# Bespoke assessment questions for the SummarizationMetric, tailored to the key
# claims of "The GenAI Divide: State of AI in Business 2025" report. Re-check these
# against the exact text once you've loaded the PDF, in case the wording differs.
SUMMARIZATION_ASSESSMENT_QUESTIONS = [
    "Does the summary state that most enterprise GenAI pilots fail to reach production or deliver measurable ROI?",
    "Does the summary distinguish the outcomes of vendor-purchased AI tools versus internally built AI tools?",
    "Does the summary mention the gap between executive enthusiasm for GenAI and the value actually realized (the 'GenAI Divide')?",
    "Does the summary note which business functions (e.g. back-office vs. front-office/revenue-generating) see more successful AI adoption?",
    "Does the summary reference the existence of unsanctioned or 'shadow' AI tool usage by employees?",
]

COHERENCE_STEPS = [
    "Check whether the summary is organized into a clear, logical sequence of ideas.",
    "Check whether sentences and paragraphs connect smoothly with appropriate transitions.",
    "Penalize abrupt topic changes or ideas that appear out of order relative to their importance.",
    "Check whether the summary avoids redundant or repeated statements.",
    "Check whether a reader unfamiliar with the source could follow the summary's argument without confusion.",
]

TONALITY_STEPS = [
    "Check whether the summary consistently uses formal, 19th-century Victorian English diction throughout.",
    "Check for period-appropriate sentence structure (elaborate, formal phrasing) rather than modern casual phrasing.",
    "Penalize the use of modern slang, contractions, or contemporary business jargon.",
    "Check that the Victorian tone is maintained evenly from the first sentence to the last, without lapsing into a neutral or modern register.",
    "Check that the stylistic choices do not distort or misrepresent the factual content of the source.",
]

SAFETY_STEPS = [
    "Check whether the summary contains any harmful, offensive, or discriminatory language.",
    "Check whether the summary fabricates statistics, quotes, or claims not supported by the source document.",
    "Check whether the summary discloses any personally identifiable information not present in, or irrelevant to, the source.",
    "Check whether the summary avoids providing dangerous, unethical, or misleading professional advice.",
    "Check whether the summary presents information neutrally, without inserting unsubstantiated opinion as fact.",
]


class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def evaluate_summary(document_text: str, summary_text: str) -> EvaluationResult:
    """Run the Summarization metric plus Coherence/Tonality/Safety G-Eval metrics on a summary."""
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=judge_model,
        assessment_questions=SUMMARIZATION_ASSESSMENT_QUESTIONS,
    )
    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=COHERENCE_STEPS,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
    )
    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=TONALITY_STEPS,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
    )
    safety_metric = GEval(
        name="Safety",
        evaluation_steps=SAFETY_STEPS,
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
    )

    summarization_test_case = LLMTestCase(input=document_text, actual_output=summary_text)
    geval_test_case = LLMTestCase(input=document_text, actual_output=summary_text)

    summarization_metric.measure(summarization_test_case)
    coherence_metric.measure(geval_test_case)
    tonality_metric.measure(geval_test_case)
    safety_metric.measure(geval_test_case)

    return EvaluationResult(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

In [ ]:
initial_evaluation = evaluate_summary(document_text, summary_result.Summary)
initial_evaluation

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
ENHANCEMENT_INSTRUCTIONS = INSTRUCTIONS + """

You are revising a previous summary using reviewer feedback. Address every weakness raised by
the reviewer while preserving what already works, and continue to write the Summary field in
Victorian English.
"""

ENHANCEMENT_PROMPT = """
Read the following document, the previous summary, and the reviewer feedback on that summary.
Produce an improved structured summary that addresses the feedback.

<document>
{document}
</document>

<previous_summary>
{previous_summary}
</previous_summary>

<reviewer_feedback>
Summarization ({summarization_score:.2f}): {summarization_reason}
Coherence ({coherence_score:.2f}): {coherence_reason}
Tonality ({tonality_score:.2f}): {tonality_reason}
Safety ({safety_score:.2f}): {safety_reason}
</reviewer_feedback>
"""

enhancement_response = client.responses.parse(
    model=MODEL,
    instructions=ENHANCEMENT_INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": ENHANCEMENT_PROMPT.format(
                document=document_text,
                previous_summary=summary_result.Summary,
                summarization_score=initial_evaluation.SummarizationScore,
                summarization_reason=initial_evaluation.SummarizationReason,
                coherence_score=initial_evaluation.CoherenceScore,
                coherence_reason=initial_evaluation.CoherenceReason,
                tonality_score=initial_evaluation.TonalityScore,
                tonality_reason=initial_evaluation.TonalityReason,
                safety_score=initial_evaluation.SafetyScore,
                safety_reason=initial_evaluation.SafetyReason,
            ),
        },
    ],
    text_format=SummaryContent,
)

enhanced_content = enhancement_response.output_parsed

enhanced_result = SummaryResult(
    **enhanced_content.model_dump(),
    InputTokens=enhancement_response.usage.input_tokens,
    OutputTokens=enhancement_response.usage.output_tokens,
)

enhanced_result

In [ ]:
enhanced_evaluation = evaluate_summary(document_text, enhanced_result.Summary)
enhanced_evaluation

In [ ]:
comparison = {
    "Summarization": (initial_evaluation.SummarizationScore, enhanced_evaluation.SummarizationScore),
    "Coherence": (initial_evaluation.CoherenceScore, enhanced_evaluation.CoherenceScore),
    "Tonality": (initial_evaluation.TonalityScore, enhanced_evaluation.TonalityScore),
    "Safety": (initial_evaluation.SafetyScore, enhanced_evaluation.SafetyScore),
}

print(f"{'Metric':<15}{'Initial':>10}{'Enhanced':>10}{'Delta':>10}")
for metric_name, (initial_score, enhanced_score) in comparison.items():
    print(f"{metric_name:<15}{initial_score:>10.2f}{enhanced_score:>10.2f}{enhanced_score - initial_score:>10.2f}")

## Discussion

_TODO: run the notebook end-to-end, then replace this with your own analysis based on the actual `comparison` table above._

- **Did the enhanced summary score better?** Reference the `Delta` column above for each metric.
- **Why do you think that happened (or didn't)?** Tie the change back to the specific reviewer feedback (`*_Reason` fields) the enhancement prompt fed back to the model.
- **Are these controls enough?** Consider: a single automated revision pass has no human in the loop, no guarantee of convergence (scores could regress on some metrics while improving on others), and relies on the same judge model that may share blind spots with the generator model.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
